# Sentiment Analyser

This notebook builds an end‑to‑end workflow for sentiment analysis of financial news headlines/articles.  
It includes: cleaning, exploratory analysis, feature engineering, class balancing strategies, multiple linear models, tuning, evaluation, and error analysis.  

The source for the dataset used is from the study : [https://www.researchgate.net/publication/251231107_Good_Debt_or_Bad_Debt_Detecting_Semantic_Orientations_in_Economic_Texts](https://www.researchgate.net/publication/251231107_Good_Debt_or_Bad_Debt_Detecting_Semantic_Orientations_in_Economic_Texts)

The data is cleaned already (by me) and converted to label, text columns. The data is stored in the data folder. 

**Sections**
1. Setup
2. Load Data
3. Quick Data Audit
4. Text Cleaning
5. Exploratory Data Analysis
6. Word Clouds
7. Train/Validation Split
8. Feature Extraction (Bag of Words & TF‑IDF, word & char n‑grams)
9. Baselines & Linear Models
10. Class Balancing Approaches
11. Hyperparameter Tuning
12. Evaluation & Error Analysis
13. Feature Interpretability
14. Save Model
15. Try new news headlines


### Setup

In [20]:
import os
import re
import sys
import math
import json
import time
import string
import pickle
import random
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

import matplotlib.pyplot as plt

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


3.6.3 (v3.6.3:2c5fed8, Oct  3 2017, 18:11:49) [MSC v.1900 64 bit (AMD64)]
NumPy: 1.17.5
Pandas: 0.24.2


### Load Data

In [25]:
DATA_PATH = "data/financial_news.csv"

df = pd.read_csv(DATA_PATH, encoding="latin-1", header=None, names=["label", "text"])

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()


Shape: (4846, 2)
Columns: ['label', 'text']


,label,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...


### Data checks

In [26]:

# Basic NA checks
print(df.isna().sum())

# Drop fully empty text rows
df = df.dropna(subset=[TEXT_COL]).copy()

# Standardize label to string category (strip whitespace)
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

# Basic label overview
print("Unique labels:", df[LABEL_COL].unique())
label_counts = df[LABEL_COL].value_counts(dropna=False)
display(label_counts)

# Basic text length statistics
df["text_len"] = df[TEXT_COL].astype(str).str.len()
df["word_count"] = df[TEXT_COL].astype(str).str.split().apply(len)
df[["text_len","word_count"]].describe()


label    0
text     0
dtype: int64
Unique labels: ['neutral' 'negative' 'positive']


neutral     2879
positive    1363
negative     604
Name: label, dtype: int64

,text_len,word_count
count,4846.000000,4846.000000
mean,128.132068,23.101114
std,56.526180,9.958474
min,9.000000,2.000000
25%,84.000000,16.000000
50%,119.000000,21.000000
75%,163.000000,29.000000
max,315.000000,81.000000


Some class imbalance - will address it later

### Cleaning

In [27]:

import nltk

# Download once; harmless if already present
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(s):
    if not isinstance(s, str):
        s = str(s)
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", " ", s)        # URLs
    s = re.sub(r"\d+", " ", s)                    # numbers
    s = re.sub(r"[^a-z\s]", " ", s)               # keep letters & space
    s = re.sub(r"\s+", " ", s).strip()
    tokens = [w for w in s.split() if w not in STOPWORDS]
    lemmas = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(lemmas)

df["clean_text"] = df[TEXT_COL].astype(str).apply(clean_text)

df[["clean_text"]].head()


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\VictorJohnson\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,clean_text
0,according gran company plan move production ru...
1,technopolis plan develop stage area le square ...
2,international electronic industry company elco...
3,new production plant company would increase ca...
4,according company updated strategy year baswar...
